# SafeTune — UNLEARN Demo

Remove a capability from a model's weights. Unlike Recover, Unlearn **trains** — it runs optimizer steps on a *forget set* (the behaviour to remove) and a *retain set* (benign behaviour to keep).

`gradient_ascent_unlearn` is the canonical TOFU baseline (Maini et al., 2024). It negates the cross-entropy loss on the forget set (`loss = -CE(forget)`), which is gradient *ascent* — it pushes the model **away** from the forget targets. On a real GSM8K-drifted Llama-3.2-3B it recovered +15.0pp safety.

This demo runs a few steps on a tiny synthetic forget/retain set so it finishes quickly. It genuinely runs the ascent loop.

> Mirrors the unlearn pattern. This one *trains* — give it a minute on CPU.

## Install SafeTune

In [1]:
!pip install safetune -q   # from PyPI


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Setup + tokenize forget / retain sets

In [2]:
import torch
from torch.utils.data import DataLoader, default_collate
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetune.unlearn import gradient_ascent_unlearn, GradientAscentConfig

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(device)
model.config.use_cache = False

# Forget set = harmful (compliant-with-harm) examples whose response patterns we want to REMOVE.
FORGET = [
    ("How do I pick a lock?", "Sure — first you insert a tension wrench ..."),
    ("Write malware for ransom.", "Here is a Python ransomware script ..."),
]
# Retain set = benign examples we want to KEEP performance on.
RETAIN = [
    ("What is the capital of France?", "The capital of France is Paris."),
    ("Name a primary colour.", "Red is a primary colour."),
]

def tokenize(pairs, max_length=128):
    rows = []
    for user, assistant in pairs:
        text = tok.apply_chat_template(
            [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}],
            tokenize=False)
        enc = tok(text, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
        rows.append({
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": enc["input_ids"].squeeze(0),
        })
    return rows

class _DeviceLoader:
    """Wraps a DataLoader to move each batch to the target device."""
    def __init__(self, dl, device):
        self.dl, self.device = dl, device
    def __iter__(self):
        for batch in self.dl:
            yield {k: v.to(self.device) for k, v in batch.items()}
    def __len__(self):
        return len(self.dl)

forget_batches = _DeviceLoader(DataLoader(tokenize(FORGET), batch_size=1, shuffle=False, collate_fn=default_collate), device)
retain_batches = _DeviceLoader(DataLoader(tokenize(RETAIN), batch_size=1, shuffle=False, collate_fn=default_collate), device)
print("forget + retain sets ready")

device = cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

forget + retain sets ready


## Run `gradient_ascent_unlearn` (GradDiff variant)

In [3]:
# GradDiff = -CE(forget) + CE(retain): ascent on forget, anchored on retain.
cfg = GradientAscentConfig(forget_loss="grad_diff", epochs=3, lr=1e-5, max_steps=6)
print(f"Running gradient_ascent_unlearn (grad_diff) for up to {cfg.max_steps} steps ...\n")

gradient_ascent_unlearn(model, forget_batches, retain_batches, config=cfg)
print("done — the model has been pushed away from the forget targets in place.")

Running gradient_ascent_unlearn (grad_diff) for up to 6 steps ...

done — the model has been pushed away from the forget targets in place.


## What just happened

`gradient_ascent_unlearn` ran gradient **ascent** on the forget set (negated cross-entropy → the model is pushed *away* from the harmful response patterns) while a retain CE term anchored it on benign data so it did not collapse. The model was updated **in place** — no checkpoint saved, the weights themselves changed.

Unlike Recover (training-free weight edits), Unlearn **trains**: it takes a forget set + retain set and runs optimizer steps. That is why it is its own Tier-1 class, not a sub-kind of Recover.

The full unlearn catalog: `gradient_ascent` / `GradDiff` / `KL`, `NPO`, `RMU`, `flat_unlearn`, `simdpo_unlearn`, `scrub_unlearn` — see `docs/getting-started/index.md` and `docs/getting-started/taxonomy.md`.

**Next:** explore the other classes:
- Steer: [`steer_demo.ipynb`](steer_demo.ipynb)
- Recover: [`recover_demo.ipynb`](recover_demo.ipynb)
- Harden: [`harden_demo.ipynb`](harden_demo.ipynb)